# Lektion 10 - Prestandaoptimering och fine-tuning
**Assignment: Transfer learning and data pipeline tuning**  

Instructions:

1. Use a pretrained model (e.g., ResNet18)
2. Compare frozen vs fine-tuned performance
3. Measure small performance tweaks

## Task 1: Transfer learning
Start with a pretrained model and a new classifier head.

In [ ]:
# TODO: Load a pretrained model
import torch
from torchvision import models 

# Vi hämtar hem resnet mha torchsvision, och hämtar vikterna från varianten
# som är tränad på ImageNets

model = models.resnet18(weights = models.ResNet18_Weights.IMAGENET1K_V1)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Lilit/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:05<00:00, 8.36MB/s]


In [4]:
# Vi tar en span på modellen

model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [2]:
# TODO: Freeze the base layers
from torch.nn import Linear
num_feats = model.fc.in_features
model.fc = Linear(num_feats, 10)

In [3]:
# Move the model to a gpu device

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model.to(device)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
# Vi behöver ett dataset att träna modellen på! 
# Vi kör CIFAR-10

from torchvision import datasets
# The downloaded data is in the form of PIL images, we need to transform them to tensors
from torchvision import transforms
transform = transforms.Compose([
    transforms.ToTensor()
])


train_data = datasets.CIFAR10(root = "data", train = True, download = True, transform = transform)
test_data = datasets.CIFAR10(root = "data", train = False, download = True, transform = transform)


# Vi behöver också en dataloader
from torch.utils.data import DataLoader

# Shuffle = True för träning, False för testning för att bibehålla ordningen på testdata
train_dataloader = DataLoader(train_data, batch_size = 32, shuffle = True)
test_dataloader = DataLoader(test_data, batch_size = 32, shuffle = False)   

100%|██████████| 170M/170M [00:10<00:00, 17.0MB/s] 
c:\Users\Lilit\Documents\GitHub\ML-Ramverk\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [6]:
# TODO: Train a new classifier head
import torch.optim as optim
import torch.nn as nn

# Istället för att stoppa in model.parameters() 
# Så stoppar vi nu bara in model.fc.parameters()
# Alltså parametrarna i vårt Fully connected layer
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Träningsloop
num_epochs = 1
for _ in range(num_epochs):
    model.train()
    for xb, yb in train_dataloader:
        # Vi ser till att flytta vår batchdata till devicen!
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()